# House Prices: BaseMLP validation strategy test

## tl;dr

- 모델과 입력 feature를 고정하고 7개 validation 관점과 3개 training-row 정책을 비교했다.
- 기존 seed-42 5-fold는 public 정책 순위를 완전히 반대로 평가했다.
- 5-fold × 3 seeds 반복 K-fold는 keep-all 0.145881, 1299 제외 0.147104,
  524·1299 제외 0.149106으로 실제 public 순위를 정확히 재현했다.
- keep-all fold 표준편차도 0.040707에서 0.026967로 낮아졌다.
- tail-balanced 4-fold도 순위를 재현했고 표준편차 0.015360으로 보조 stress test에 적합했다.
- adversarial OOF AUC는 0.5139로 강한 train/test 구분 신호가 없었다.
- 권장안은 반복 K-fold를 주 검증으로, tail-balanced 4-fold를 보조 검증으로 사용하는 것이다.

## Context & Methods

목표는 local CV를 Kaggle public 숫자에 직접 맞추는 것이 아니라, 검증된 세 training-row
정책의 public 순위를 더 안정적으로 재현하는 validation 설계를 찾는 것이다.

### Key Assumptions

- 가격 모델은 역사적 BaseMLP recipe로 고정하며 생성 feature는 추가하지 않는다.
- 비교 정책은 keep-all, Id=1299 training 제외, Id=524·1299 training 제외다.
- validation 행은 어떤 정책에서도 삭제하지 않는다.
- public 점수는 기존 실제 제출 결과이며 순위 진단에만 사용한다.
- test target은 없고 adversarial holdout은 test predictor만 사용하는 transductive 보조 진단이다.
- 추천 후보는 반복 검증 또는 구조적 tail 균형 방법 중에서 고른다.

## Data

data/train.csv와 data/test.csv를 사용한다. test는 adversarial validation의
target-free 분포 비교에만 사용하며 SalePrice는 존재하지 않는다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "validation_test" /
    "basemlp_validation_strategy_comparison.ipynb"
)
REPORT_DIR = ROOT / "reports" / "validation_test"
ARTIFACT_DIR = ROOT / "artifacts" / "validation_test"
FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_validation_fold_results.csv"
STRATEGY_RESULTS_PATH = REPORT_DIR / "basemlp_validation_strategy_results.csv"
RANK_RESULTS_PATH = REPORT_DIR / "basemlp_validation_rank_alignment.csv"
SUMMARY_PATH = REPORT_DIR / "basemlp_validation_strategy_summary.json"
RUN_ID = "VALTEST-20260719-BASEMLP-NB-01"
PUBLIC_PROBE_SCORES = {
    "keep_all": 0.12579,
    "exclude_1299": 0.13016,
    "exclude_524_1299": 0.13426,
}
POLICIES = {
    "keep_all": (),
    "exclude_1299": (1299,),
    "exclude_524_1299": (524, 1299),
}
SPLIT_SEEDS = (42, 2026, 3407)

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"test: {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("external project script imports: 0")

train: 1,460 rows × 81 columns
test: 1,459 rows × 80 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0


### Fixed model input

기존 public probe와 비교 가능하도록 역사적 BaseMLP 전처리를 self-contained로
재현한다. 가격 모델 입력과 architecture는 모든 validation 전략에서 동일하다.

In [2]:
def clean_rows_without_record_correction(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned

X = clean_rows_without_record_correction(
    train.drop(columns="SalePrice"), categorical_columns
)
X_test = clean_rows_without_record_correction(
    test, categorical_columns
)
assert float(
    X.loc[X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2007
assert list(X_test.columns) == [
    column for column in X.columns
]
print("Historical BaseMLP preprocessing reproduced; model features unchanged.")


def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

Historical BaseMLP preprocessing reproduced; model features unchanged.


## Validation strategies

기본/반복 K-fold, 타깃 분위수 균형, tail 균형, adversarial test-like holdout,
Id 순서 forward holdout을 구성하고 각 split의 tail 분포를 점검한다.

In [3]:
def target_quantile_bins(
    target_log: np.ndarray,
    bins: int = 10,
) -> np.ndarray:
    return pd.qcut(
        pd.Series(target_log).rank(method="first"),
        q=bins,
        labels=False,
    ).to_numpy(dtype=int)


def repeated_kfold_splits() -> list[dict[str, object]]:
    rows = []
    for seed in SPLIT_SEEDS:
        splitter = KFold(
            n_splits=5, shuffle=True, random_state=seed
        )
        for fold, (train_index, valid_index) in enumerate(
            splitter.split(X), start=1
        ):
            rows.append({
                "base_strategy": "repeated_kfold5_3seeds",
                "component_seed": seed,
                "fold": fold,
                "train_index": train_index,
                "valid_index": valid_index,
            })
    return rows


def repeated_target_quantile_splits() -> list[dict[str, object]]:
    rows = []
    bins = target_quantile_bins(y_log, bins=10)
    for seed in SPLIT_SEEDS:
        splitter = StratifiedKFold(
            n_splits=5, shuffle=True, random_state=seed
        )
        for fold, (train_index, valid_index) in enumerate(
            splitter.split(X, bins), start=1
        ):
            rows.append({
                "base_strategy": "repeated_target_quantile5_3seeds",
                "component_seed": seed,
                "fold": fold,
                "train_index": train_index,
                "valid_index": valid_index,
            })
    return rows


def tail_balanced_4fold_splits() -> list[dict[str, object]]:
    area = pd.to_numeric(train["GrLivArea"], errors="raise")
    tail_index = np.flatnonzero(area.to_numpy() > 4000)
    non_tail_index = np.flatnonzero(area.to_numpy() <= 4000)
    assert len(tail_index) == 4

    non_tail_bins = target_quantile_bins(
        y_log[non_tail_index], bins=10
    )
    splitter = StratifiedKFold(
        n_splits=4, shuffle=True, random_state=SEED
    )
    rng = np.random.default_rng(SEED)
    shuffled_tail = tail_index.copy()
    rng.shuffle(shuffled_tail)
    rows = []
    for fold, (non_tail_train, non_tail_valid) in enumerate(
        splitter.split(non_tail_index, non_tail_bins), start=1
    ):
        valid_index = np.concatenate([
            non_tail_index[non_tail_valid],
            np.asarray([shuffled_tail[fold - 1]]),
        ])
        train_index = np.setdiff1d(
            np.arange(len(X)), valid_index, assume_unique=False
        )
        rows.append({
            "base_strategy": "tail_balanced4_seed42",
            "component_seed": SEED,
            "fold": fold,
            "train_index": train_index,
            "valid_index": valid_index,
        })
    return rows


def adversarial_testlike_holdout() -> tuple[
    list[dict[str, object]], dict[str, float | int]
]:
    combined = pd.concat([X, X_test], ignore_index=True)
    domain_target = np.concatenate([
        np.zeros(len(X), dtype=int),
        np.ones(len(X_test), dtype=int),
    ])
    preprocessor = make_preprocessor(list(NUMERIC_COLUMNS))
    encoded = preprocessor.fit_transform(combined).astype(np.float32)
    classifier = LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver="liblinear",
        random_state=SEED,
    )
    cv = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=SEED
    )
    probability = cross_val_predict(
        classifier,
        encoded,
        domain_target,
        cv=cv,
        method="predict_proba",
    )[:, 1]
    auc = float(roc_auc_score(domain_target, probability))
    train_probability = probability[:len(X)]
    holdout_size = len(X) // 5
    valid_index = np.argsort(train_probability)[-holdout_size:]
    train_index = np.setdiff1d(
        np.arange(len(X)), valid_index, assume_unique=False
    )
    return ([{
        "base_strategy": "adversarial_testlike_holdout20",
        "component_seed": SEED,
        "fold": 1,
        "train_index": train_index,
        "valid_index": valid_index,
    }], {
        "oof_domain_auc": auc,
        "holdout_rows": int(len(valid_index)),
        "mean_test_probability_in_holdout": float(
            train_probability[valid_index].mean()
        ),
        "mean_test_probability_in_training": float(
            train_probability[train_index].mean()
        ),
    })


def forward_id_holdout() -> list[dict[str, object]]:
    order = np.argsort(
        pd.to_numeric(train["Id"], errors="raise").to_numpy()
    )
    holdout_size = len(X) // 5
    valid_index = order[-holdout_size:]
    train_index = order[:-holdout_size]
    return [{
        "base_strategy": "forward_id_holdout20",
        "component_seed": SEED,
        "fold": 1,
        "train_index": train_index,
        "valid_index": valid_index,
    }]


adversarial_splits, adversarial_diagnostic = (
    adversarial_testlike_holdout()
)
base_strategy_splits = {
    "repeated_kfold5_3seeds": repeated_kfold_splits(),
    "repeated_target_quantile5_3seeds": (
        repeated_target_quantile_splits()
    ),
    "tail_balanced4_seed42": tail_balanced_4fold_splits(),
    "adversarial_testlike_holdout20": adversarial_splits,
    "forward_id_holdout20": forward_id_holdout(),
}

tail_mask = (
    pd.to_numeric(train["GrLivArea"], errors="raise")
    .to_numpy() > 4000
)
split_audit_rows = []
for strategy, splits in base_strategy_splits.items():
    for split in splits:
        valid_index = split["valid_index"]
        split_audit_rows.append({
            "base_strategy": strategy,
            "component_seed": split["component_seed"],
            "fold": split["fold"],
            "train_rows": len(split["train_index"]),
            "validation_rows": len(valid_index),
            "validation_tail_rows": int(
                tail_mask[valid_index].sum()
            ),
            "validation_log_target_mean": float(
                y_log[valid_index].mean()
            ),
            "validation_log_target_std": float(
                y_log[valid_index].std(ddof=1)
            ),
        })
split_audit = pd.DataFrame(split_audit_rows)
display(
    split_audit.groupby("base_strategy").agg(
        splits=("fold", "size"),
        validation_rows_min=("validation_rows", "min"),
        validation_rows_max=("validation_rows", "max"),
        tail_rows_min=("validation_tail_rows", "min"),
        tail_rows_max=("validation_tail_rows", "max"),
        log_target_mean_std=(
            "validation_log_target_mean", "std"
        ),
    ).round(6)
)
display(pd.Series(adversarial_diagnostic, name="adversarial diagnostic"))


,splits,validation_rows_min,validation_rows_max,tail_rows_min,tail_rows_max,log_target_mean_std
base_strategy,,,,,,
adversarial_testlike_holdout20,1,292,292,0,0,NaN
forward_id_holdout20,1,292,292,2,2,NaN
repeated_kfold5_3seeds,15,292,292,0,2,0.020293
repeated_target_quantile5_3seeds,15,292,292,0,2,0.007788
tail_balanced4_seed42,4,365,365,1,1,0.003960


oof_domain_auc                         0.513905
holdout_rows                         292.000000
mean_test_probability_in_holdout       0.705958
mean_test_probability_in_training      0.444240
Name: adversarial diagnostic, dtype: float64

## Results

각 validation 전략과 training-row 정책 조합에서 fold 전처리와 BaseMLP를 새로
적합하고 best checkpoint를 복원해 RMSLE를 계산한다.

In [4]:
if FOLD_RESULTS_PATH.exists():
    fold_results = pd.read_csv(FOLD_RESULTS_PATH)
    print("Loaded existing executed fold results; training skipped.")
    display(
        fold_results.groupby(
            ["base_strategy", "policy"]
        )["rmsle"].agg(["mean", "std", "count"]).round(6)
    )
else:
    def evaluate_strategy_policy(
        strategy: str,
        splits: list[dict[str, object]],
        policy: str,
        excluded_ids: tuple[int, ...],
    ) -> list[dict[str, object]]:
        experiment_id = (
            f"{RUN_ID}-{strategy}-{policy}"
        )
        output_dir = ARTIFACT_DIR / strategy / policy
        checkpoint_dir = output_dir / "checkpoints"
        preprocessor_dir = output_dir / "preprocessors"
        epoch_log_dir = output_dir / "logs"
        for directory in (
            checkpoint_dir, preprocessor_dir, epoch_log_dir
        ):
            directory.mkdir(parents=True, exist_ok=False)

        rows = []
        for split_number, split in enumerate(splits, start=1):
            train_index = np.asarray(
                split["train_index"], dtype=int
            )
            valid_index = np.asarray(
                split["valid_index"], dtype=int
            )
            if excluded_ids:
                keep_mask = ~train.iloc[train_index]["Id"].isin(
                    excluded_ids
                ).to_numpy()
                candidate_train_index = train_index[keep_mask]
            else:
                candidate_train_index = train_index

            fold_seed = (
                int(split["component_seed"])
                + int(split["fold"])
            )
            checkpoint_path = (
                checkpoint_dir /
                f"split_{split_number}_best.pt"
            )
            preprocessor_path = (
                preprocessor_dir /
                f"split_{split_number}_preprocessor.joblib"
            )
            epoch_log_path = (
                epoch_log_dir /
                f"split_{split_number}_epochs.csv"
            )

            preprocessor = make_preprocessor(list(NUMERIC_COLUMNS))
            X_train = preprocessor.fit_transform(
                X.iloc[candidate_train_index]
            ).astype(np.float32)
            X_valid = preprocessor.transform(
                X.iloc[valid_index]
            ).astype(np.float32)
            joblib.dump(preprocessor, preprocessor_path)

            _, training = train_fold(
                X_train,
                y_log[candidate_train_index],
                X_valid,
                y_log[valid_index],
                seed=fold_seed,
                experiment_id=experiment_id,
                fold=split_number,
                checkpoint_path=checkpoint_path,
                epoch_log_path=epoch_log_path,
            )
            rows.append({
                "base_strategy": strategy,
                "policy": policy,
                "excluded_ids": "|".join(
                    str(value) for value in excluded_ids
                ),
                "component_seed": int(split["component_seed"]),
                "fold": int(split["fold"]),
                "split_number": split_number,
                "training_rows": int(len(candidate_train_index)),
                "validation_rows": int(len(valid_index)),
                "validation_tail_rows": int(
                    tail_mask[valid_index].sum()
                ),
                "rmsle": float(
                    training["restored_validation_rmsle"]
                ),
                "best_epoch": int(training["best_epoch"]),
                "stopping_epoch": int(
                    training["stopping_epoch"]
                ),
                "encoded_features": int(X_train.shape[1]),
                "checkpoint_path": str(
                    checkpoint_path.relative_to(ROOT)
                ),
            })
        print(
            f"{strategy} / {policy}: "
            f"{np.mean([row['rmsle'] for row in rows]):.6f}"
        )
        return rows


    fold_rows = []
    for strategy, splits in base_strategy_splits.items():
        for policy, excluded_ids in POLICIES.items():
            fold_rows.extend(
                evaluate_strategy_policy(
                    strategy, splits, policy, excluded_ids
                )
            )

    fold_results = pd.DataFrame(fold_rows)
    fold_results.to_csv(FOLD_RESULTS_PATH, index=False)
    display(
        fold_results.groupby(
            ["base_strategy", "policy"]
        )["rmsle"].agg(["mean", "std", "count"]).round(6)
    )


Loaded existing executed fold results; training skipped.


mean       std  count
base_strategy                    policy                                     
adversarial_testlike_holdout20   exclude_1299      0.146956       NaN      1
                                 exclude_524_1299  0.146827       NaN      1
                                 keep_all          0.145420       NaN      1
forward_id_holdout20             exclude_1299      0.154705       NaN      1
                                 exclude_524_1299  0.179222       NaN      1
                                 keep_all          0.154705       NaN      1
repeated_kfold5_3seeds           exclude_1299      0.147104  0.026552     15
                                 exclude_524_1299  0.149106  0.033177     15
                                 keep_all          0.145881  0.026967     15
repeated_target_quantile5_3seeds exclude_1299      0.141947  0.025418     15
                                 exclude_524_1299  0.147452  0.036324     15
                                 keep_all          0.142605  0.021809     15
tail_balanced4_seed42            exclude_1299      0.145137  0.018099      4
                                 exclude_524_1299  0.151584  0.028041      4
                                 keep_all          0.144592  0.015360      4

In [5]:
aggregation_specs = {
    "kfold5_seed42": (
        (fold_results["base_strategy"]
         == "repeated_kfold5_3seeds")
        & (fold_results["component_seed"] == 42)
    ),
    "repeated_kfold5_3seeds": (
        fold_results["base_strategy"]
        == "repeated_kfold5_3seeds"
    ),
    "target_quantile5_seed42": (
        (fold_results["base_strategy"]
         == "repeated_target_quantile5_3seeds")
        & (fold_results["component_seed"] == 42)
    ),
    "repeated_target_quantile5_3seeds": (
        fold_results["base_strategy"]
        == "repeated_target_quantile5_3seeds"
    ),
    "tail_balanced4_seed42": (
        fold_results["base_strategy"]
        == "tail_balanced4_seed42"
    ),
    "adversarial_testlike_holdout20": (
        fold_results["base_strategy"]
        == "adversarial_testlike_holdout20"
    ),
    "forward_id_holdout20": (
        fold_results["base_strategy"]
        == "forward_id_holdout20"
    ),
}
strategy_meta = {
    "kfold5_seed42": {
        "folds": "5",
        "seeds": "42",
        "methodological_priority": 5,
        "uses_test_covariates": False,
    },
    "repeated_kfold5_3seeds": {
        "folds": "5x3",
        "seeds": "42|2026|3407",
        "methodological_priority": 2,
        "uses_test_covariates": False,
    },
    "target_quantile5_seed42": {
        "folds": "5",
        "seeds": "42",
        "methodological_priority": 4,
        "uses_test_covariates": False,
    },
    "repeated_target_quantile5_3seeds": {
        "folds": "5x3",
        "seeds": "42|2026|3407",
        "methodological_priority": 1,
        "uses_test_covariates": False,
    },
    "tail_balanced4_seed42": {
        "folds": "4",
        "seeds": "42",
        "methodological_priority": 3,
        "uses_test_covariates": False,
    },
    "adversarial_testlike_holdout20": {
        "folds": "holdout",
        "seeds": "42",
        "methodological_priority": 6,
        "uses_test_covariates": True,
    },
    "forward_id_holdout20": {
        "folds": "holdout",
        "seeds": "42",
        "methodological_priority": 7,
        "uses_test_covariates": False,
    },
}

strategy_rows = []
for strategy, mask in aggregation_specs.items():
    subset = fold_results.loc[mask]
    for policy in POLICIES:
        policy_scores = subset.loc[
            subset["policy"].eq(policy), "rmsle"
        ].to_numpy(dtype=np.float64)
        assert len(policy_scores) > 0
        strategy_rows.append({
            "strategy": strategy,
            "policy": policy,
            "local_score": float(policy_scores.mean()),
            "fold_std": (
                float(policy_scores.std(ddof=1))
                if len(policy_scores) > 1 else np.nan
            ),
            "score_count": int(len(policy_scores)),
            "public_probe_score": PUBLIC_PROBE_SCORES[policy],
            "absolute_public_gap": float(abs(
                policy_scores.mean()
                - PUBLIC_PROBE_SCORES[policy]
            )),
            **strategy_meta[strategy],
        })
strategy_results = pd.DataFrame(strategy_rows)
strategy_results["local_rank"] = (
    strategy_results.groupby("strategy")["local_score"]
    .rank(method="average", ascending=True)
)
strategy_results["public_rank"] = (
    strategy_results.groupby("strategy")[
        "public_probe_score"
    ].rank(method="average", ascending=True)
)
strategy_results.to_csv(STRATEGY_RESULTS_PATH, index=False)

rank_rows = []
public_order = list(
    pd.Series(PUBLIC_PROBE_SCORES)
    .sort_values().index
)
rank_tolerance = 1e-12
for strategy, group in strategy_results.groupby(
    "strategy", sort=False
):
    score_by_policy = (
        group.set_index("policy")["local_score"]
        .loc[list(POLICIES.keys())]
    )
    local_order = list(
        score_by_policy.sort_values().index
    )
    tied_pairs = [
        f"{left}={right}"
        for left_index, left in enumerate(POLICIES)
        for right in list(POLICIES)[left_index + 1:]
        if abs(
            float(score_by_policy[left])
            - float(score_by_policy[right])
        ) <= rank_tolerance
    ]
    exact_public_order = (
        float(score_by_policy["keep_all"])
        < float(score_by_policy["exclude_1299"])
        - rank_tolerance
        and float(score_by_policy["exclude_1299"])
        < float(score_by_policy["exclude_524_1299"])
        - rank_tolerance
    )
    keep_score = float(score_by_policy["keep_all"])
    other_best = min(
        float(score_by_policy["exclude_1299"]),
        float(score_by_policy["exclude_524_1299"]),
    )
    keep_all_selected = (
        keep_score < other_best - rank_tolerance
    )
    keep_all_best_or_tied = (
        keep_score <= other_best + rank_tolerance
    )
    rank_correlation = float(np.corrcoef(
        group.set_index("policy").loc[
            list(POLICIES.keys()), "local_rank"
        ],
        group.set_index("policy").loc[
            list(POLICIES.keys()), "public_rank"
        ],
    )[0, 1])
    rank_rows.append({
        "strategy": strategy,
        "local_policy_order": " > ".join(local_order),
        "local_policy_ties": "|".join(tied_pairs),
        "public_policy_order": " > ".join(public_order),
        "exact_public_order": exact_public_order,
        "keep_all_selected": keep_all_selected,
        "keep_all_best_or_tied": keep_all_best_or_tied,
        "rank_correlation": rank_correlation,
        "mean_absolute_public_gap": float(
            group["absolute_public_gap"].mean()
        ),
        "keep_all_local_score": keep_score,
        "keep_all_fold_std": float(
            group.loc[
                group["policy"].eq("keep_all"),
                "fold_std",
            ].iloc[0]
        ),
        **strategy_meta[strategy],
    })
rank_results = pd.DataFrame(rank_rows)

eligible = rank_results.loc[
    rank_results["strategy"].isin([
        "repeated_target_quantile5_3seeds",
        "repeated_kfold5_3seeds",
        "tail_balanced4_seed42",
    ])
].copy()
eligible["selection_score"] = (
    eligible["exact_public_order"].astype(int) * 100
    + eligible["keep_all_selected"].astype(int) * 30
    + eligible["rank_correlation"] * 10
    - eligible["methodological_priority"]
)
recommended_strategy = str(
    eligible.sort_values(
        ["selection_score", "keep_all_fold_std"],
        ascending=[False, True],
    ).iloc[0]["strategy"]
)
rank_results["recommended"] = (
    rank_results["strategy"].eq(recommended_strategy)
)
rank_results.to_csv(RANK_RESULTS_PATH, index=False)

display(
    strategy_results.pivot(
        index="strategy",
        columns="policy",
        values="local_score",
    ).round(6)
)
display(
    rank_results.sort_values(
        ["recommended", "selection_score"]
        if "selection_score" in rank_results.columns
        else ["recommended", "rank_correlation"],
        ascending=False,
    ).round(6)
)
print(f"recommended validation strategy: {recommended_strategy}")


policy,exclude_1299,exclude_524_1299,keep_all
strategy,,,
adversarial_testlike_holdout20,0.146956,0.146827,0.145420
forward_id_holdout20,0.154705,0.179222,0.154705
kfold5_seed42,0.150128,0.148096,0.150342
repeated_kfold5_3seeds,0.147104,0.149106,0.145881
repeated_target_quantile5_3seeds,0.141947,0.147452,0.142605
tail_balanced4_seed42,0.145137,0.151584,0.144592
target_quantile5_seed42,0.146123,0.155451,0.144119


,strategy,local_policy_order,local_policy_ties,public_policy_order,exact_public_order,keep_all_selected,keep_all_best_or_tied,rank_correlation,mean_absolute_public_gap,keep_all_local_score,keep_all_fold_std,folds,seeds,methodological_priority,uses_test_covariates,recommended
1,repeated_kfold5_3seeds,keep_all > exclude_1299 > exclude_524_1299,,keep_all > exclude_1299 > exclude_524_1299,True,True,True,1.000000,0.017294,0.145881,0.026967,5x3,42|2026|3407,2,False,True
2,target_quantile5_seed42,keep_all > exclude_1299 > exclude_524_1299,,keep_all > exclude_1299 > exclude_524_1299,True,True,True,1.000000,0.018494,0.144119,0.020542,5,42,4,False,False
4,tail_balanced4_seed42,keep_all > exclude_1299 > exclude_524_1299,,keep_all > exclude_1299 > exclude_524_1299,True,True,True,1.000000,0.017034,0.144592,0.015360,4,42,3,False,False
6,forward_id_holdout20,keep_all > exclude_1299 > exclude_524_1299,keep_all=exclude_1299,keep_all > exclude_1299 > exclude_524_1299,False,False,True,0.866025,0.032807,0.154705,NaN,holdout,42,7,False,False
3,repeated_target_quantile5_3seeds,exclude_1299 > keep_all > exclude_524_1299,,keep_all > exclude_1299 > exclude_524_1299,False,False,False,0.500000,0.013931,0.142605,0.021809,5x3,42|2026|3407,1,False,False
5,adversarial_testlike_holdout20,keep_all > exclude_524_1299 > exclude_1299,,keep_all > exclude_1299 > exclude_524_1299,False,True,True,0.500000,0.016331,0.145420,NaN,holdout,42,6,True,False
0,kfold5_seed42,exclude_524_1299 > exclude_1299 > keep_all,,keep_all > exclude_1299 > exclude_524_1299,False,False,False,-1.000000,0.019452,0.150342,0.040707,5,42,5,False,False


recommended validation strategy: repeated_kfold5_3seeds


## Records

fold 결과, 전략별 집계, public 순위 정렬 진단과 실험 로그를 저장한다.

In [6]:
run_timestamp = datetime.now(timezone.utc).isoformat()
for row in strategy_results.to_dict(orient="records"):
    strategy = row["strategy"]
    policy = row["policy"]
    strategy_rank = rank_results.loc[
        rank_results["strategy"].eq(strategy)
    ].iloc[0]
    matching_folds = fold_results.loc[
        aggregation_specs[strategy]
        & fold_results["policy"].eq(policy)
    ]
    experiment_id = (
        f"{RUN_ID}-{strategy}-{policy}"
    ).upper().replace("_", "-")
    append_experiment({
        "experiment_id": experiment_id,
        "datetime": run_timestamp,
        "objective": "Compare BaseMLP validation strategies without adding model features",
        "data_version": (
            f"train sha256={sha256(TRAIN_PATH)}; "
            f"test sha256={sha256(TEST_PATH)}"
        ),
        "split_strategy": strategy,
        "folds": str(row["folds"]),
        "seed": str(row["seeds"]),
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": "Historical BaseMLP preprocessing fixed; fold median/scaling/one-hot; no new features; exclusions apply only to fold training",
        "features": "79 original predictors; Id excluded; no generated features",
        "model": "BaseMLP",
        "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
        "optimizer": "Adam",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "learning_rate": str(LEARNING_RATE),
        "batch_size": str(BATCH_SIZE),
        "max_epochs": str(MAX_EPOCHS),
        "patience": str(PATIENCE),
        "best_epoch": "splits=" + "|".join(
            str(value)
            for value in matching_folds["best_epoch"]
        ),
        "hyperparameters": json.dumps({
            "policy": policy,
            "excluded_training_ids": list(POLICIES[policy]),
            "historical_public_probe_score": (
                PUBLIC_PROBE_SCORES[policy]
            ),
            "public_score_is_not_from_this_split_strategy": True,
            "uses_test_covariates": bool(
                row["uses_test_covariates"]
            ),
            "external_script_dependency": False,
        }, ensure_ascii=False),
        "cv_mean": str(row["local_score"]),
        "cv_std": (
            "" if pd.isna(row["fold_std"])
            else str(row["fold_std"])
        ),
        "checkpoint_path": str(
            (ARTIFACT_DIR /
             matching_folds.iloc[0]["base_strategy"] /
             policy / "checkpoints").relative_to(ROOT)
        ),
        "artifact_path": "; ".join([
            str(NOTEBOOK_PATH.relative_to(ROOT)),
            str(FOLD_RESULTS_PATH.relative_to(ROOT)),
            str(STRATEGY_RESULTS_PATH.relative_to(ROOT)),
            str(RANK_RESULTS_PATH.relative_to(ROOT)),
            str(SUMMARY_PATH.relative_to(ROOT)),
        ]),
        "result": "validation_strategy_diagnostic",
        "interpretation": (
            f"Policy={policy}; public-rank correlation="
            f"{strategy_rank['rank_correlation']:+.3f}; "
            f"keep_all_selected="
            f"{bool(strategy_rank['keep_all_selected'])}."
        ),
        "next_action": (
            f"Use {recommended_strategy} as the primary "
            "validation candidate and retain current KFold "
            "as a comparability reference."
        ),
    })

summary_experiment_id = (
    f"{RUN_ID}-RANK-RECONCILIATION"
).upper().replace("_", "-")
append_experiment({
    "experiment_id": summary_experiment_id,
    "datetime": run_timestamp,
    "objective": "Select a validation strategy using repeated stability and historical public policy-rank alignment",
    "data_version": (
        f"train sha256={sha256(TRAIN_PATH)}; "
        f"test sha256={sha256(TEST_PATH)}"
    ),
    "split_strategy": recommended_strategy,
    "folds": str(
        strategy_meta[recommended_strategy]["folds"]
    ),
    "seed": str(
        strategy_meta[recommended_strategy]["seeds"]
    ),
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "Validation-only diagnostic; historical BaseMLP preprocessing and features fixed",
    "features": "79 original predictors; Id excluded; no generated features",
    "model": "BaseMLP",
    "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
    "optimizer": "Adam",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": str(LEARNING_RATE),
    "batch_size": str(BATCH_SIZE),
    "max_epochs": str(MAX_EPOCHS),
    "patience": str(PATIENCE),
    "hyperparameters": json.dumps({
        "candidate_strategies": list(aggregation_specs),
        "historical_public_probe_scores": PUBLIC_PROBE_SCORES,
        "public_scores_used_for_split_generation": False,
        "rank_tolerance": rank_tolerance,
        "external_script_dependency": False,
    }, ensure_ascii=False),
    "artifact_path": "; ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        str(STRATEGY_RESULTS_PATH.relative_to(ROOT)),
        str(RANK_RESULTS_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
    ]),
    "result": "recommended_validation_strategy",
    "interpretation": (
        f"Recommended={recommended_strategy}; selection "
        "prioritizes repeated/structured validation, unique "
        "keep-all selection, and historical public rank alignment."
    ),
    "next_action": "Use the recommended strategy as primary and retain seed-42 KFold only as a historical comparability reference.",
})

forward_tie_correction_id = (
    f"{RUN_ID}-FORWARD-HOLDOUT-TIE-CORRECTION"
).upper().replace("_", "-")
append_experiment({
    "experiment_id": forward_tie_correction_id,
    "datetime": run_timestamp,
    "objective": "Correct tie handling in the forward-Id holdout diagnostic",
    "data_version": f"train sha256={sha256(TRAIN_PATH)}",
    "split_strategy": "forward_id_holdout20",
    "folds": "holdout",
    "seed": str(SEED),
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "No model rerun; append-only rank reconciliation from executed fold results",
    "features": "79 original predictors; Id excluded; no generated features",
    "model": "BaseMLP",
    "artifact_path": "; ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        str(RANK_RESULTS_PATH.relative_to(ROOT)),
        str(SUMMARY_PATH.relative_to(ROOT)),
    ]),
    "result": "diagnostic_tie_correction",
    "interpretation": "keep_all and exclude_1299 are exactly tied at 0.1547047597494557; forward holdout cannot uniquely rank these policies.",
    "next_action": "Do not use forward-Id holdout as the primary validation design.",
})

summary = {
    "run_id": RUN_ID,
    "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
    "scope": {
        "model_features_changed": False,
        "model_recipe_changed": False,
        "validation_only": True,
        "external_script_dependency": False,
    },
    "source": {
        "train_path": "data/train.csv",
        "test_path": "data/test.csv",
        "train_sha256": sha256(TRAIN_PATH),
        "test_sha256": sha256(TEST_PATH),
    },
    "public_probe_scores": PUBLIC_PROBE_SCORES,
    "public_probe_caveat": (
        "Historical Kaggle public scores diagnose ranking "
        "alignment only; they were not produced by every "
        "candidate split strategy."
    ),
    "adversarial_validation": adversarial_diagnostic,
    "recommended_strategy": recommended_strategy,
    "strategy_rank_alignment": rank_results.to_dict(
        orient="records"
    ),
    "strategy_policy_scores": strategy_results.to_dict(
        orient="records"
    ),
    "artifacts": {
        "fold_results": str(
            FOLD_RESULTS_PATH.relative_to(ROOT)
        ),
        "strategy_results": str(
            STRATEGY_RESULTS_PATH.relative_to(ROOT)
        ),
        "rank_results": str(
            RANK_RESULTS_PATH.relative_to(ROOT)
        ),
    },
    "validation": {
        "notebook_executed_top_to_bottom": True,
        "public_scores_used_for_final_model_tuning": False,
        "test_target_available": False,
        "kaggle_private_scores": "unverified",
    },
}
SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        default=lambda value: (
            value.item() if hasattr(value, "item")
            else str(value)
        ),
    ),
    encoding="utf-8",
)
print(f"summary saved: {SUMMARY_PATH.relative_to(ROOT)}")


summary saved: reports/validation_test/basemlp_validation_strategy_summary.json


## Takeaways

- 주 검증: KFold(5, shuffle=True)를 split seed 42, 2026, 3407에서 반복하고 15개 fold를 평균한다.
- 역사적 비교를 위해 seed-42 단독 점수도 계속 병기하되 모델 선택에는 단독 사용하지 않는다.
- 보조 검증: GrLivArea > 4,000 네 행을 validation fold마다 하나씩 배치한 tail-balanced 4-fold를 사용한다.
- target-quantile 단일 seed는 public 순위를 재현했지만 반복 시 1299 제외를 근소하게 선호해 주 검증으로 채택하지 않는다.
- adversarial holdout은 AUC 0.5139라 test-like 분할 근거가 약하고, forward holdout은 정책 동률과 큰 score gap 때문에 채택하지 않는다.
- public 점수는 split 생성이나 모델 학습에 사용하지 않았고 정책 순위 진단에만 사용했다.
- 생성 feature와 가격 모델 recipe 변경은 0건이며 외부 프로젝트 스크립트 의존도 0건이다.